# Stage 20B: disjoint-well confirmation

Stage 20Aの診断で最も安定していたweight 0.05、cap 8 ftを固定し、Stage 20Aで使った158 wellsを完全除外した別sampleで確認します。このNotebookの結果を見てweightを変更せず、全gate通過時だけStage 20Cへ進みます。Kaggle提出は作りません。CPUランタイムを使用してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir():
    subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else:
    subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
assert (data_dir/'train').is_dir(),data_dir
def run_checked(command):
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout,flush=True)
    if result.returncode:
        print(result.stderr,flush=True); raise RuntimeError(f'command failed: {command}')


## 固定artifactとdiscovery exclusion

Stage 20Aの`cut_features.parquet`からdiscovery wellsを読み、候補poolから先に除外します。cut単位ではなくwell単位の完全除外です。


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
stage20a_run=artifact_dir/'stage20a_top_pf_alignment_full_v001'
assert (stage16b_run/'pseudo_test_manifest.parquet').is_file(),stage16b_run
assert (stage17a_run/'replay_predictions.parquet').is_file(),stage17a_run
assert (stage20a_run/'cut_features.parquet').is_file(),stage20a_run
stage20a=json.loads((stage20a_run/'summary.json').read_text())
assert stage20a['stage20a_complete'] is True and stage20a['sample_wells']==158
print(stage16b_run,stage17a_run,stage20a_run,sep='\n')


## 独立well confirmation

A130/SP45 proxyの計算量はStage 20Aと同程度です。25 cutsごとに進捗を表示します。


In [ ]:
RUN_ID='stage20b_disjoint_confirmation_full_v001'; run_dir=artifact_dir/RUN_ID
if not (run_dir/'summary.json').is_file():
    run_checked(['uv','run','rogii-trajectory-base-alignment','--config','configs/experiment/stage20b_disjoint_confirmation.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--exclude-run',str(stage20a_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID])
summary=json.loads((run_dir/'summary.json').read_text())
{
 'stage20b_complete':summary['stage20b_complete'], 'promoted_to_stage20c':summary['promoted_to_stage20c'], 'sample_cuts':summary['sample_cuts'],'sample_wells':summary['sample_wells'], 'excluded_discovery_wells':summary['excluded_discovery_wells'], 'discovery_well_overlap':summary['discovery_well_overlap'], 'base_comparison':summary['base_comparison'], 'standard_base_rmse':summary['standard_report']['base_rmse'], 'standard_candidate_rmse':summary['standard_report']['candidate_rmse'], 'standard_delta':summary['standard_report']['rmse_delta'], 'bootstrap_95pct':summary['bootstrap_95pct'],'gates':summary['gates'], 'next_step':summary['next_step'],
}


In [ ]:
import pandas as pd
pd.read_parquet(run_dir/'weight_report.parquet').sort_values('weight')

最後の辞書とweight表を共有してください。昇格判定はweight 0.05だけで完結しており、診断表の別weightへ変更してStage 20Bを再実行しません。
